## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Mount Google Drive and save all trained-model outputs there immediately.
- Download only the exact code, data, and pre-computed structural-feature files needed into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.
- Avoid `git clone`, which can be less reliable in hosted notebook runtimes.

Required GitHub-hosted artifacts:
- `models/baseline_frozen_esm2.pt`
- `data/ss3/deepalgpro_train_ss3_structured.json.gz` / `deepalgpro_test_ss3_structured.json.gz`
- `data/disorder/deepalgpro_train_disorder.json.gz` / `deepalgpro_test_disorder.json.gz`

Section C reads `results/rsa_regularization/sweep_summary.csv` produced by notebook 12.
On Colab this is read from Google Drive; run notebook 12 first and keep Drive mounted.

In [3]:
import os
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
else:
    DRIVE_ROOT = None

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR = RUNTIME_ROOT / 'src'
PACKAGE_DIR = SRC_DIR / 'xallergen'
DATA_DIR = RUNTIME_ROOT / 'data'
SS3_DIR = DATA_DIR / 'ss3'
DISORDER_DIR = DATA_DIR / 'disorder'
RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
RUNTIME_RESULTS_DIR = RUNTIME_ROOT / 'results'
HF_DIR = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

if IS_COLAB:
    MODELS_DIR = DRIVE_ROOT / 'models'
    RESULTS_DIR = DRIVE_ROOT / 'results'
else:
    MODELS_DIR = RUNTIME_MODELS_DIR
    RESULTS_DIR = RUNTIME_RESULTS_DIR

for path in [
    SRC_DIR, PACKAGE_DIR, DATA_DIR, SS3_DIR, DISORDER_DIR,
    RUNTIME_MODELS_DIR, RUNTIME_RESULTS_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    package_files = [
        '__init__.py',
        'baseline_notebook_utils.py',
        'baseline_sasa_experiment.py',
        'mtl_epitope_notebook_utils.py',
        'rsa_preprocessing.py',
    ]
    data_files = [
        'deepalgpro_train_cleaned.csv',
        'deepalgpro_test_cleaned.csv',
        'positives_splitB.csv',
    ]
    ss3_files = [
        'deepalgpro_train_ss3_structured.json.gz',
        'deepalgpro_test_ss3_structured.json.gz',
    ]
    disorder_files = [
        'deepalgpro_train_disorder.json.gz',
        'deepalgpro_test_disorder.json.gz',
    ]

    download_jobs = []
    download_jobs.extend((f'{RAW}/src/xallergen/{name}', PACKAGE_DIR / name) for name in package_files)
    download_jobs.extend((f'{RAW}/data/{name}', DATA_DIR / name) for name in data_files)
    download_jobs.extend((f'{RAW}/data/ss3/{name}', SS3_DIR / name) for name in ss3_files)
    download_jobs.extend((f'{RAW}/data/disorder/{name}', DISORDER_DIR / name) for name in disorder_files)
    download_jobs.append((f'{RAW}/models/baseline_frozen_esm2.pt', RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'))

    failed_urls = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed_urls.append((url, str(exc)))

    if failed_urls:
        details = '\n'.join(f'  - {url} -> {err}' for url, err in failed_urls)
        raise RuntimeError('Failed to download required GitHub artifacts:\n' + details)

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'SS3 dir: {SS3_DIR}')
print(f'Disorder dir: {DISORDER_DIR}')
print(f'Output results dir: {RESULTS_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

RUN_TARGET: colab
Runtime root: /content/XAllergen2.0
SS3 dir: /content/XAllergen2.0/data/ss3
Disorder dir: /content/XAllergen2.0/data/disorder
Output results dir: /content/drive/MyDrive/XAllergen2.0/results


# Section A — SS3-Structured Regularization Sweep

Tests whether penalising attention on coil/loop residues improves allergenicity classification
and residue-level localisation. The regularization loss is:

$$\mathcal{L} = \lambda_\text{cls}\mathcal{L}_\text{cls} + \lambda_\text{reg} \cdot \frac{1}{L}\sum_i \alpha_i (1 - f_i)$$

where $f_i = 1.0$ for helix (H) or strand (E) residues and $f_i = 0.0$ for coil/loop (C).
The $(1 - f_i)$ term is therefore 0 for structured residues and 1 for coil residues,
so the loss penalises attention weight on unstructured positions.

No new model heads or trainable components are introduced; the frozen ESM-2 backbone is unchanged.
Section C compares the best SS3-structured λ against the RSA baseline from notebook 12.

In [4]:
import importlib
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() or (candidate / 'src' / 'xallergen').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing pyproject.toml or src/xallergen')


if 'DATA_DIR' not in globals():
    RUNTIME_ROOT = _find_repo_root(Path.cwd())
    SRC_DIR = RUNTIME_ROOT / 'src'
    DATA_DIR = RUNTIME_ROOT / 'data'
    SS3_DIR = DATA_DIR / 'ss3'
    DISORDER_DIR = DATA_DIR / 'disorder'
    RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
    RESULTS_DIR = RUNTIME_ROOT / 'results'
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

missing_modules = []
for module_name in ('numpy', 'pandas', 'torch', 'IPython', 'sklearn'):
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_modules.append(module_name)

if missing_modules:
    missing = ', '.join(missing_modules)
    raise ModuleNotFoundError(
        'Notebook dependencies are missing from the active kernel: '
        f'{missing}. Run `make setup` and select `.venv/bin/python` or the '
        '`Python (xallergen2)` kernel before rerunning this cell.'
    )

import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import (
    RANDOM_STATE,
    seed_everything,
)
from xallergen.baseline_sasa_experiment import (
    RSARegularizationConfig,
    inspect_rsa_inputs,
    load_rsa_lookup_dicts,
    run_lambda_rsa_sweep,
)
from xallergen.rsa_preprocessing import compute_rsa_ss3_structured_correlation

TRAIN_CSV = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV = DATA_DIR / 'deepalgpro_test_cleaned.csv'
POSITIVES_SPLITB_CSV = DATA_DIR / 'positives_splitB.csv'
BASELINE_CHECKPOINT_PATH = RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'
TRAIN_SS3_PATH = SS3_DIR / 'deepalgpro_train_ss3_structured.json.gz'
TEST_SS3_PATH = SS3_DIR / 'deepalgpro_test_ss3_structured.json.gz'
TRAIN_RSA_PATH = DATA_DIR / 'rsa' / 'deepalgpro_train_rsa.json.gz'
SS3_RESULTS_DIR = RESULTS_DIR / 'ss3_regularization'
SS3_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SS3_SWEEP_CSV = SS3_RESULTS_DIR / 'sweep_summary.csv'

for required_path in [
    TRAIN_CSV, TEST_CSV, TRAIN_SS3_PATH, TEST_SS3_PATH,
    POSITIVES_SPLITB_CSV, BASELINE_CHECKPOINT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

# λ=0 (unregularized baseline) is skipped — already present in the RSA sweep
# and identical across constraints. It appears in Section C for comparison.
SS3_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
    feature_key='ss3_structured',
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Feature: {SS3_CONFIG.feature_key}  (1.0=H/E structured, 0.0=C coil)')
print(f'λ values: {SS3_CONFIG.lambda_rsa_values}')
print(f'Output dir: {SS3_RESULTS_DIR}')

Device: cuda
Feature: ss3_structured  (1.0=H/E structured, 0.0=C coil)
λ values: (0.1, 0.5, 1.0, 5.0)
Output dir: /content/drive/MyDrive/XAllergen2.0/results/ss3_regularization


In [5]:
ss3_train_df = pd.read_csv(TRAIN_CSV).copy()
ss3_test_df = pd.read_csv(TEST_CSV).copy()
for df in [ss3_train_df, ss3_test_df]:
    df['sequence_id'] = df['sequence_id'].astype(str).str.strip()
    df['sequence'] = df['sequence'].astype(str).str.strip().str.upper()
    df['label'] = df['label'].astype(int)

# rsa_min/rsa_max reflect the binary SS3-structured indicator (0.0 or 1.0)
display(inspect_rsa_inputs(
    train_rsa_path=TRAIN_SS3_PATH,
    test_rsa_path=TEST_SS3_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
))

,path,format,compressed,n_sequences,rsa_min,rsa_max,rsa_in_unit_interval,min_length,max_length,expected_sequences,missing_sequences,extra_sequences,exact_id_match
0,/content/XAllergen2.0/data/ss3/deepalgpro_trai...,precomputed_rsa_json,True,5551,0.0,1.0,True,11,999,5551,0,0,True
1,/content/XAllergen2.0/data/ss3/deepalgpro_test...,precomputed_rsa_json,True,1377,0.0,1.0,True,11,979,1377,0,0,True


In [6]:
seed_everything(RANDOM_STATE)
ss3_train_split_df, ss3_val_split_df = train_test_split(
    ss3_train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=ss3_train_df['label'],
)
ss3_train_split_df = ss3_train_split_df.reset_index(drop=True).copy()
ss3_val_split_df = ss3_val_split_df.reset_index(drop=True).copy()

print(f'Train: {len(ss3_train_split_df):,} | Val: {len(ss3_val_split_df):,} | Test: {len(ss3_test_df):,}')

Train: 4,995 | Val: 556 | Test: 1,377


In [7]:
ss3_train_lookup, ss3_test_lookup, ss3_lookup_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_SS3_PATH,
    test_rsa_path=TEST_SS3_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
    add_special_tokens=SS3_CONFIG.add_special_tokens,
)
display(pd.DataFrame([ss3_lookup_summary['train'], ss3_lookup_summary['test']]))

,path,format,compressed,n_expected_sequences,n_parsed_sequences,n_missing_sequences,n_extra_sequences,n_length_mismatches,missing_sequence_ids,extra_sequence_ids,length_mismatches,add_special_tokens
0,/content/XAllergen2.0/data/ss3/deepalgpro_trai...,precomputed_rsa_json,True,5551,5551,0,0,0,[],[],[],False
1,/content/XAllergen2.0/data/ss3/deepalgpro_test...,precomputed_rsa_json,True,1377,1377,0,0,0,[],[],[],False


In [8]:
import gzip, json

if not TRAIN_RSA_PATH.exists():
    print(f'Skipping RSA-vs-SS3 correlation; missing optional file: {TRAIN_RSA_PATH}')
else:
    with gzip.open(TRAIN_RSA_PATH, 'rt') as f:
        rsa_train_lookup_raw = json.load(f)

    # Pearson r between RSA and SS3-structured pooled over all train residues.
    # Low |r| means the two constraints carry complementary signal.
    compute_rsa_ss3_structured_correlation(rsa_train_lookup_raw, ss3_train_lookup)

Skipping RSA-vs-SS3 correlation; missing optional file: /content/XAllergen2.0/data/rsa/deepalgpro_train_rsa.json.gz


In [9]:
ss3_sweep_df = run_lambda_rsa_sweep(
    config=SS3_CONFIG,
    train_split_df=ss3_train_split_df,
    val_split_df=ss3_val_split_df,
    test_df=ss3_test_df,
    train_rsa_lookup=ss3_train_lookup,
    test_rsa_lookup=ss3_test_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=SS3_RESULTS_DIR,
    device=DEVICE,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
ss3_sweep_df.to_csv(SS3_SWEEP_CSV, index=False)
display(ss3_sweep_df)
print(f'Saved to: {SS3_SWEEP_CSV}')
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lambda_rsa=0.1:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_cls=0.43261 | train_rsa=0.00417 | train_total=0.43303 | val_cls=0.33749 | val_rsa=0.00385 | val_total=0.33787 | val_auroc=0.94350 | best=1
Epoch   2/30 | train_cls=0.31302 | train_rsa=0.00440 | train_total=0.31346 | val_cls=0.27910 | val_rsa=0.00398 | val_total=0.27950 | val_auroc=0.95716 | best=2
Epoch   3/30 | train_cls=0.27820 | train_rsa=0.00447 | train_total=0.27865 | val_cls=0.27015 | val_rsa=0.00402 | val_total=0.27055 | val_auroc=0.96120 | best=3
Epoch   4/30 | train_cls=0.25634 | train_rsa=0.00451 | train_total=0.25679 | val_cls=0.25019 | val_rsa=0.00405 | val_total=0.25059 | val_auroc=0.96513 | best=4
Epoch   5/30 | train_cls=0.23931 | train_rsa=0.00453 | train_total=0.23976 | val_cls=0.25327 | val_rsa=0.00407 | val_total=0.25368 | val_auroc=0.96280 | best=4
Epoch   6/30 | train_cls=0.22990 | train_rsa=0.00455 | train_total=0.23036 | val_cls=0.23666 | val_rsa=0.00409 | val_total=0.23707 | val_auroc=0.96826 | best=6
Epoch   7/30 | train_cls=0.22005 | train

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

...

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating proteins:   0%|          | 0/61 [00:00<?, ?it/s]

Evaluating proteins: processed 5/61 proteins
Evaluating proteins: processed 10/61 proteins
Evaluating proteins: processed 15/61 proteins
Evaluating proteins: processed 20/61 proteins
Evaluating proteins: processed 25/61 proteins
Evaluating proteins: processed 30/61 proteins
Evaluating proteins: processed 35/61 proteins
Evaluating proteins: processed 40/61 proteins
Evaluating proteins: processed 45/61 proteins
Evaluating proteins: processed 50/61 proteins
Evaluating proteins: processed 55/61 proteins
Evaluating proteins: processed 60/61 proteins
Evaluating proteins: processed 61/61 proteins


,lambda_rsa,epoch,val_auroc,val_precision,val_recall,val_f1,val_mcc,val_accuracy,test_auroc,test_precision,test_recall,test_f1,test_mcc,test_accuracy,residue_auroc,residue_auprc,residue_precision_at_k
0,0.1,16,0.977064,0.916667,0.926740,0.921676,0.845353,0.922662,0.968984,0.899425,0.931548,0.915205,0.832067,0.915759,0.4699,0.2537,0.2155
1,0.5,21,0.979329,0.933579,0.926740,0.930147,0.863270,0.931655,0.969759,0.916542,0.915179,0.915860,0.835776,0.917938,0.4732,0.2553,0.2221
2,1.0,21,0.979989,0.933086,0.919414,0.926199,0.856120,0.928058,0.969744,0.919643,0.919643,0.919643,0.843047,0.921569,0.4789,0.2583,0.2267
3,5.0,24,0.984170,0.934545,0.941392,0.937956,0.877697,0.938849,0.968442,0.914831,0.927083,0.920916,0.844621,0.922295,0.5300,0.2863,0.2649


Saved to: /content/drive/MyDrive/XAllergen2.0/results/ss3_regularization/sweep_summary.csv


In [10]:
best_ss3 = ss3_sweep_df.loc[ss3_sweep_df['val_mcc'].idxmax()]
print('=== SS3-structured: best λ by val MCC ===')
print(f'  Best λ       : {best_ss3["lambda_rsa"]}')
print(f'  Val MCC      : {best_ss3["val_mcc"]:.4f}')
print(f'  Test MCC     : {best_ss3["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best_ss3["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best_ss3["residue_auroc"]:.4f}')

=== SS3-structured: best λ by val MCC ===
  Best λ       : 5.0
  Val MCC      : 0.8777
  Test MCC     : 0.8446
  Test AUROC   : 0.9684
  Residue AUROC: 0.5300


---
## Section B — Disorder Regularization Sweep

Tests whether penalising attention on ordered (low-disorder) residues improves allergenicity
classification. The same loss form as Section A, with per-residue disorder score as $f_i$:

$$\mathcal{L} = \lambda_\text{cls}\mathcal{L}_\text{cls} + \lambda_\text{reg} \cdot \frac{1}{L}\sum_i \alpha_i (1 - f_i)$$

where $f_i = 1.0$ for disordered residues and $f_i = 0.0$ for ordered residues.
The $(1 - f_i)$ term is 0 for disordered positions and 1 for ordered positions,
so the loss penalises attention on structured/ordered residues — reflecting that allergen
epitopes tend to localise in flexible, disordered regions.

In [11]:
TRAIN_DISORDER_PATH = DISORDER_DIR / 'deepalgpro_train_disorder.json.gz'
TEST_DISORDER_PATH = DISORDER_DIR / 'deepalgpro_test_disorder.json.gz'
DISORDER_RESULTS_DIR = RESULTS_DIR / 'disorder_regularization'
DISORDER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DISORDER_SWEEP_CSV = DISORDER_RESULTS_DIR / 'sweep_summary.csv'

DISORDER_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
    feature_key='disorder',
)

for required_path in [TRAIN_DISORDER_PATH, TEST_DISORDER_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

seed_everything(RANDOM_STATE)
print(f'Feature: {DISORDER_CONFIG.feature_key}  (1.0=disordered, 0.0=ordered)')
print(f'λ values: {DISORDER_CONFIG.lambda_rsa_values}')
print(f'Output dir: {DISORDER_RESULTS_DIR}')

Feature: disorder  (1.0=disordered, 0.0=ordered)
λ values: (0.1, 0.5, 1.0, 5.0)
Output dir: /content/drive/MyDrive/XAllergen2.0/results/disorder_regularization


In [12]:
display(inspect_rsa_inputs(
    train_rsa_path=TRAIN_DISORDER_PATH,
    test_rsa_path=TEST_DISORDER_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
))

,path,format,compressed,n_sequences,rsa_min,rsa_max,rsa_in_unit_interval,min_length,max_length,expected_sequences,missing_sequences,extra_sequences,exact_id_match
0,/content/XAllergen2.0/data/disorder/deepalgpro...,precomputed_rsa_json,True,5551,0.000007,0.999941,True,11,999,5551,0,0,True
1,/content/XAllergen2.0/data/disorder/deepalgpro...,precomputed_rsa_json,True,1377,0.000008,0.999861,True,11,979,1377,0,0,True


In [13]:
disorder_train_lookup, disorder_test_lookup, disorder_lookup_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_DISORDER_PATH,
    test_rsa_path=TEST_DISORDER_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
    add_special_tokens=DISORDER_CONFIG.add_special_tokens,
)
display(pd.DataFrame([disorder_lookup_summary['train'], disorder_lookup_summary['test']]))

,path,format,compressed,n_expected_sequences,n_parsed_sequences,n_missing_sequences,n_extra_sequences,n_length_mismatches,missing_sequence_ids,extra_sequence_ids,length_mismatches,add_special_tokens
0,/content/XAllergen2.0/data/disorder/deepalgpro...,precomputed_rsa_json,True,5551,5551,0,0,0,[],[],[],False
1,/content/XAllergen2.0/data/disorder/deepalgpro...,precomputed_rsa_json,True,1377,1377,0,0,0,[],[],[],False


In [14]:
disorder_sweep_df = run_lambda_rsa_sweep(
    config=DISORDER_CONFIG,
    train_split_df=ss3_train_split_df,
    val_split_df=ss3_val_split_df,
    test_df=ss3_test_df,
    train_rsa_lookup=disorder_train_lookup,
    test_rsa_lookup=disorder_test_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=DISORDER_RESULTS_DIR,
    device=DEVICE,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
disorder_sweep_df.to_csv(DISORDER_SWEEP_CSV, index=False)
display(disorder_sweep_df)
print(f'Saved to: {DISORDER_SWEEP_CSV}')
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lambda_rsa=0.1:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_cls=0.43253 | train_rsa=0.00444 | train_total=0.43297 | val_cls=0.33747 | val_rsa=0.00414 | val_total=0.33788 | val_auroc=0.94358 | best=1
Epoch   2/30 | train_cls=0.31309 | train_rsa=0.00425 | train_total=0.31352 | val_cls=0.27883 | val_rsa=0.00405 | val_total=0.27924 | val_auroc=0.95709 | best=2
Epoch   3/30 | train_cls=0.27857 | train_rsa=0.00420 | train_total=0.27899 | val_cls=0.27024 | val_rsa=0.00399 | val_total=0.27064 | val_auroc=0.96113 | best=3
Epoch   4/30 | train_cls=0.25711 | train_rsa=0.00418 | train_total=0.25753 | val_cls=0.24999 | val_rsa=0.00401 | val_total=0.25039 | val_auroc=0.96517 | best=4
Epoch   5/30 | train_cls=0.24000 | train_rsa=0.00418 | train_total=0.24042 | val_cls=0.25329 | val_rsa=0.00401 | val_total=0.25369 | val_auroc=0.96275 | best=4
Epoch   6/30 | train_cls=0.22976 | train_rsa=0.00416 | train_total=0.23018 | val_cls=0.23695 | val_rsa=0.00398 | val_total=0.23735 | val_auroc=0.96809 | best=6
Epoch   7/30 | train_cls=0.21929 | train

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

...

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating proteins:   0%|          | 0/61 [00:00<?, ?it/s]

Evaluating proteins: processed 5/61 proteins
Evaluating proteins: processed 10/61 proteins
Evaluating proteins: processed 15/61 proteins
Evaluating proteins: processed 20/61 proteins
Evaluating proteins: processed 25/61 proteins
Evaluating proteins: processed 30/61 proteins
Evaluating proteins: processed 35/61 proteins
Evaluating proteins: processed 40/61 proteins
Evaluating proteins: processed 45/61 proteins
Evaluating proteins: processed 50/61 proteins
Evaluating proteins: processed 55/61 proteins
Evaluating proteins: processed 60/61 proteins
Evaluating proteins: processed 61/61 proteins


,lambda_rsa,epoch,val_auroc,val_precision,val_recall,val_f1,val_mcc,val_accuracy,test_auroc,test_precision,test_recall,test_f1,test_mcc,test_accuracy,residue_auroc,residue_auprc,residue_precision_at_k
0,0.1,16,0.977957,0.916968,0.930403,0.923636,0.848999,0.924460,0.969660,0.901004,0.934524,0.917458,0.836475,0.917938,0.4689,0.2534,0.2159
1,0.5,16,0.977012,0.923358,0.926740,0.925046,0.852485,0.926259,0.969637,0.899281,0.930060,0.914411,0.830567,0.915033,0.4680,0.2529,0.2162
2,1.0,16,0.976935,0.916968,0.930403,0.923636,0.848999,0.924460,0.969841,0.901004,0.934524,0.917458,0.836475,0.917938,0.4653,0.2515,0.2168
3,5.0,16,0.977660,0.920290,0.930403,0.925319,0.852549,0.926259,0.969225,0.892045,0.934524,0.912791,0.826694,0.912854,0.4523,0.2448,0.2112


Saved to: /content/drive/MyDrive/XAllergen2.0/results/disorder_regularization/sweep_summary.csv


In [15]:
best_disorder = disorder_sweep_df.loc[disorder_sweep_df['val_mcc'].idxmax()]
print('=== Disorder: best λ by val MCC ===')
print(f'  Best λ       : {best_disorder["lambda_rsa"]}')
print(f'  Val MCC      : {best_disorder["val_mcc"]:.4f}')
print(f'  Test MCC     : {best_disorder["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best_disorder["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best_disorder["residue_auroc"]:.4f}')

=== Disorder: best λ by val MCC ===
  Best λ       : 5.0
  Val MCC      : 0.8525
  Test MCC     : 0.8267
  Test AUROC   : 0.9692
  Residue AUROC: 0.4523


---
## Section C — Comparison Table

- **Baseline (λ=0)** — unregularized model, read from the RSA sweep's λ=0 entry.
- **RSA** — best λ > 0 by val MCC from notebook 12.
- **Disorder** — best λ > 0 by val MCC from Section B above.
- **SS3-structured** — best λ > 0 by val MCC from Section A above.

In [16]:
RSA_SWEEP_CSV = RESULTS_DIR / 'rsa_regularization' / 'sweep_summary.csv'
if not RSA_SWEEP_CSV.exists():
    raise FileNotFoundError(
        f'RSA sweep summary not found at {RSA_SWEEP_CSV}. '
        'Run notebook 12 first and ensure its output CSV is present in the results directory.'
    )

def _row(csv_path, label, select_best=True):
    df = pd.read_csv(csv_path)
    row = df.loc[df['val_mcc'].idxmax()] if select_best else df.loc[df['lambda_rsa'].eq(0.0)].iloc[0]
    return {
        'Constraint': label,
        'Best λ': row['lambda_rsa'],
        'Val MCC': round(float(row['val_mcc']), 4),
        'Test MCC': round(float(row['test_mcc']), 4),
        'Test AUROC': round(float(row['test_auroc']), 4),
        'Residue AUROC': round(float(row['residue_auroc']), 4),
    }

comparison_df = pd.DataFrame([
    _row(RSA_SWEEP_CSV,      'Baseline (λ=0)',  select_best=False),
    _row(RSA_SWEEP_CSV,      'RSA',             select_best=True),
    _row(DISORDER_SWEEP_CSV, 'Disorder',        select_best=True),
    _row(SS3_SWEEP_CSV,      'SS3-structured',  select_best=True),
]).set_index('Constraint')

print('=== Comparison (λ>0 rows selected by val MCC) ===')
display(comparison_df)

=== Comparison (λ>0 rows selected by val MCC) ===


,Best λ,Val MCC,Test MCC,Test AUROC,Residue AUROC
Constraint,,,,,
Baseline (λ=0),0.0,0.8454,0.8336,0.9699,0.4677
RSA,0.5,0.8597,0.8474,0.9697,0.4660
Disorder,5.0,0.8525,0.8267,0.9692,0.4523
SS3-structured,5.0,0.8777,0.8446,0.9684,0.5300


In [ ]:
# Shut down the runtime after everything above has finished.
os.kill(os.getpid(), 9)

: 

: 

: 